In [ ]:
import torch
import torch.nn.functional as F

def vae_loss_function(reconstructed_x, original_x, mu, logvar):
    # 1. Reconstruction Loss (BCE): Check karta hai pixels kitne sahi bane
    # reduction='sum' karne se saare pixels ka loss ek sath jadd jata hai
    BCE = F.binary_cross_entropy(reconstructed_x, original_x, reduction='sum')

    # 2. KL Divergence Loss: Check karta hai latent space bell curve jaisa hai ya nahi
    # Yeh exact mathematical formula hai KLD ka VAE ke liye
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Total Loss dono ka jod hota hai
    return BCE + KLD

In [ ]:
import torch
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim=128):
        super(VAE, self).__init__()

        # === 1. ENCODER LAYERS ===
        self.conv1 = nn.Conv2d(3, 32, 4, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 4, stride=2, padding=1)

        self.flatten = nn.Flatten()

        # Latent Space (Split) Layers
        self.fc_mu = nn.Linear(128 * 8 * 8, latent_dim)
        self.fc_logvar = nn.Linear(128 * 8 * 8, latent_dim)

        # === 2. DECODER LAYERS (Tera waala style) ===
        self.fc_init = nn.Linear(latent_dim, 128 * 8 * 8)
        self.m1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.m2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.m3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)

        # Activations
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def forward(self, x):
        # Encoder Flow (Beech mein ReLU zaroori hai!)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x_flat = self.flatten(x)

        mu = self.fc_mu(x_flat)
        logvar = self.fc_logvar(x_flat)

        # Reparameterization
        z = self.reparameterize(mu, logvar)

        # Decoder Flow
        dec_x = self.relu(self.fc_init(z))
        dec_x = dec_x.view(-1, 128, 8, 8)

        dec_x = self.relu(self.m1(dec_x))
        dec_x = self.relu(self.m2(dec_x))
        reconstructed_img = self.sigmoid(self.m3(dec_x)) # Output range 0 se 1 ho gayi

        return reconstructed_img, mu, logvar

# === AB MODEL INITIALIZE KARO ===
vae_model = VAE(latent_dim=128)
print("Main VAE Model successfully create aur initialize ho gaya hai!")

Main VAE Model successfully create aur initialize ho gaya hai!


In [ ]:
import torch.optim as optim

# 1. Device set kiya (Agar GPU available hai toh GPU par chalega, nahi toh CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Model ko device par bheja aur Optimizer set kiya
vae_model = vae_model.to(device)
optimizer = optim.Adam(vae_model.parameters(), lr=1e-3)

# 3. Training Loop (Maan lo hum 10 epochs ke liye chalayein)
num_epochs = 10

for epoch in range(num_epochs):
    vae_model.train() # Model ko training mode mein dala
    train_loss = 0

    for batch_idx, (images, _) in enumerate(train_dataloader):
        # Images ko GPU/CPU par bheja aur 0-1 ke beech normalize kiya (agar pehle se nahi hai)
        images = images.to(device)

        # Gradients ko zero kiya
        optimizer.zero_grad()

        # Forward Pass: Model se teeno cheezein nikali
        reconstructed_images, mu, logvar = vae_model(images)

        # Loss calculate kiya
        loss = vae_loss_function(reconstructed_images, images, mu, logvar)

        # Backward Pass: Galti piche bheji
        loss.backward()

        # Weights ko update kiya
        optimizer.step()

        train_loss += loss.item()

    # Epoch khatam hone par average loss print kiya
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss / len(train_dataloader.dataset):.4f}")

Epoch [1/10], Loss: 6438.4955
Epoch [2/10], Loss: 6289.9965
Epoch [3/10], Loss: 6269.4297
Epoch [4/10], Loss: 6260.9575
Epoch [5/10], Loss: 6255.8516
Epoch [6/10], Loss: 6252.3235
Epoch [7/10], Loss: 6249.7633
Epoch [8/10], Loss: 6247.9767
Epoch [9/10], Loss: 6246.1922
Epoch [10/10], Loss: 6245.0424
